# Pakistan Sign Language (PSL) Alphabet Recognizer
**Course**: AL-2002 Artificial Intelligence (Laboratory), Fall 2026  
**Student**: Areesha Ahmad (24K-3043)  
**Project Notebook**: End-to-end data processing, feature normalisation, 20-seed experiment evaluation, and agent demonstration.

## 1. Environment and Library Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from src.features import HandSample, load_dataset
from src.models import BaselineRecognizer, KNNRecognizer, SVMRecognizer, ForestRecognizer
from src.evaluate import run_experiments
from src.agents import SignAgent, WordBuilder
from src.config import RAW_DATA_PATH

print('All modules imported successfully!')

## 2. Dataset Overview
Let us inspect the recorded primary dataset (`data/raw/psl_data.csv`).
It contains 64 columns: 63 raw 3D landmark coordinates plus 1 ground-truth sign label.

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)
print(f'Total recorded samples: {len(df)}')
print('\nSamples per class:')
print(df['label'].value_counts())
df.head(3)

## 3. Geometric Feature Normalisation
Using `HandSample`, raw coordinates are normalized in two steps:
1. **Translation**: The wrist is shifted to `(0, 0, 0)`.
2. **Scale**: Coordinates are divided by the maximum distance from the wrist.
The wrist is discarded, leaving **60 scale- and position-invariant numbers**.

In [ ]:
X, y = load_dataset(RAW_DATA_PATH)
print(f'Normalised feature matrix shape (X): {X.shape}')
print(f'Target labels vector shape (y): {y.shape}')
print('First 6 normalised features of Sample 0:')
print(np.round(X[0][:6], 4))

## 4. Experimental Evaluation (20 Seeded Splits)
We evaluate all four models over 20 independent seeded splits to evaluate mean accuracy, Macro F1, training time, and per-sample prediction speed.

In [ ]:
df_summary, raw_results = run_experiments(X, y)
df_summary

## 5. Visual Results and Charts
Displaying the three generated evaluation charts.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

img1 = Image.open('outputs/figures/model_accuracy_comparison.png')
axes[0].imshow(img1)
axes[0].axis('off')
axes[0].set_title('Accuracy Comparison', fontsize=12)

img2 = Image.open('outputs/figures/confusion_matrix.png')
axes[1].imshow(img2)
axes[1].axis('off')
axes[1].set_title('Confusion Matrix (SVM)', fontsize=12)

img3 = Image.open('outputs/figures/learning_curve.png')
axes[2].imshow(img3)
axes[2].axis('off')
axes[2].set_title('Learning Curve', fontsize=12)

plt.tight_layout()
plt.show()

## 6. Model-Based Agent and WordBuilder Demonstration
Demonstrating the 10-frame smoothing memory and the 15-frame word builder locking letters into conjoined Urdu script.

In [ ]:
svm = SVMRecognizer()
svm.fit(X, y)
agent = SignAgent(model=svm)
wb = WordBuilder(hold_frames=15)

# Simulate holding 'bay' for 15 frames
for _ in range(15):
    wb.update('bay')

# Simulate holding 'laam' for 15 frames
for _ in range(15):
    wb.update('laam')

print(f'Spelled Roman Word : {wb.get_word()}')
print(f'Spelled Native Urdu: {wb.get_urdu_word()} (بل)')